In [15]:
import sys
import os

# Add parent directory to path so we can import src
sys.path.insert(0, os.path.abspath('..'))

from src.ingest import *
from src.profiling import *

listings = load_listings("../data/raw/London/listings.csv")
calendar = load_calendar("../data/raw/London/calendar.csv")
reviews = load_reviews("../data/raw/London/reviews.csv")

print(profile_dataframe(listings, "listings"))
print(profile_dataframe(calendar, "calendar"))
print(profile_dataframe(reviews, "reviews"))



{'dataset': 'listings', 'rows': 96871, 'columns': 79, 'missing_values': np.int64(1017701), 'duplicate_rows': np.int64(0)}
{'dataset': 'calendar', 'rows': 35357974, 'columns': 7, 'missing_values': np.int64(70715948), 'duplicate_rows': np.int64(0)}
{'dataset': 'reviews', 'rows': 2097996, 'columns': 6, 'missing_values': np.int64(203), 'duplicate_rows': np.int64(0)}


In [16]:
# Check for duplicates
import importlib
import sys
sys.path.insert(0, '..')

import src.profiling
importlib.reload(src.profiling)
from src.profiling import check_duplicates

check_duplicates(listings)
check_duplicates(calendar)
check_duplicates(reviews)

Duplicate Rows: 0
Duplicate Rows: 0
Duplicate Rows: 0


np.int64(0)

In [18]:
# Convert price to numeric (remove $ and commas)
listings['price_numeric'] = pd.to_numeric(
    listings['price'].astype(str).str.replace('$', '').str.replace(',', ''),
    errors='coerce'
)

price_outliers = detect_outliers_iqr(
    listings,
    "price_numeric"
)

print(
    len(price_outliers)
)

4195


In [19]:
# Detect outliers for availability_365
availability_outliers = detect_outliers_iqr(
    listings,
    "availability_365"
)

print(f"Availability 365 outliers: {len(availability_outliers)}")

# Detect outliers for number_of_reviews
reviews_count_outliers = detect_outliers_iqr(
    listings,
    "number_of_reviews"
)

print(f"Number of reviews outliers: {len(reviews_count_outliers)}")

Availability 365 outliers: 0
Number of reviews outliers: 11446


In [6]:
import sys
import os
import importlib
sys.path.insert(0, os.path.abspath('..'))

from src.ingest import load_listings
import src.validation
importlib.reload(src.validation)
from src.validation import *

listings = load_listings("../data/raw/London/listings.csv")

print(
    len(
        validate_price(
            listings
        )
    )
)

print(
    len(
        validate_latitude(
            listings
        )
    )
)

0
0


In [14]:
import sys
import os
import importlib
sys.path.insert(0, os.path.abspath('..'))

import src.cleaning
importlib.reload(src.cleaning)
from src.cleaning import *
from src.ingest import load_listings, load_calendar, load_reviews

listings = load_listings("../data/raw/London/listings.csv")
calendar = load_calendar("../data/raw/London/calendar.csv")
reviews = load_reviews("../data/raw/London/reviews.csv")

listings = clean_price(listings)

listings = convert_dates(
    listings,
    "host_since"
)

calendar = convert_dates(
    calendar,
    "date"
)

reviews = convert_dates(
    reviews,
    "date"
)

listings = standardize_text(
    listings,
    "room_type"
)


#Missing Value Imputation ("review_scores_rating")
print("Review Scores Rating - Before imputation:")
print(f"Missing values: {listings['review_scores_rating'].isna().sum()}")
print(f"Mean: {listings['review_scores_rating'].mean():.2f}")

median_rating = listings[
    "review_scores_rating"
].median()

listings[
    "review_scores_rating"
] = listings[
    "review_scores_rating"
].fillna(
    median_rating
)

print(f"After imputation - Missing values: {listings['review_scores_rating'].isna().sum()}\n")

# Missing Value Imputation ("review_scores_location")
print("Review Scores Location - Before imputation:")
print(f"Missing values: {listings['review_scores_location'].isna().sum()}")
print(f"Mean: {listings['review_scores_location'].mean():.2f}")

median_location = listings[
    "review_scores_location"
].median()

listings[
    "review_scores_location"
] = listings[
    "review_scores_location"
].fillna(
    median_location
)

print(f"After imputation - Missing values: {listings['review_scores_location'].isna().sum()}\n")

# Missing Value Imputation ("review_scores_value")
print("Review Scores Value - Before imputation:")
print(f"Missing values: {listings['review_scores_value'].isna().sum()}")
print(f"Mean: {listings['review_scores_value'].mean():.2f}")

median_value = listings[
    "review_scores_value"
].median()

listings[
    "review_scores_value"
] = listings[
    "review_scores_value"
].fillna(
    median_value
)

print(f"After imputation - Missing values: {listings['review_scores_value'].isna().sum()}\n")

# Missing Value Imputation ("review_scores_cleanliness")
print("Review Scores Cleanliness - Before imputation:")
print(f"Missing values: {listings['review_scores_cleanliness'].isna().sum()}")
print(f"Mean: {listings['review_scores_cleanliness'].mean():.2f}")

median_cleanliness = listings[
    "review_scores_cleanliness"
].median()

listings[
    "review_scores_cleanliness"
] = listings[
    "review_scores_cleanliness"
].fillna(
    median_cleanliness
)
#Drop 100% Missing Columns
listings = listings.drop(
    columns=[
        "license",
        "calendar_updated",
        "neighbourhood_group_cleansed"
    ],
    errors="ignore"
)

print(f"After imputation - Missing values: {listings['review_scores_cleanliness'].isna().sum()}")

import os

os.makedirs(
    "../data/processed",
    exist_ok=True
)

listings.to_csv(
    "../data/processed/listings_clean.csv",
    index=False
)

calendar.to_csv(
    "../data/processed/calendar_clean.csv",
    index=False
)

reviews.to_csv(
    "../data/processed/reviews_clean.csv",
    index=False
)

print("Cleaned files saved successfully!")

Review Scores Rating - Before imputation:
Missing values: 24122
Mean: 4.68
After imputation - Missing values: 0

Review Scores Location - Before imputation:
Missing values: 24166
Mean: 4.73
After imputation - Missing values: 0

Review Scores Value - Before imputation:
Missing values: 24166
Mean: 4.62
After imputation - Missing values: 0

Review Scores Cleanliness - Before imputation:
Missing values: 24131
Mean: 4.65
After imputation - Missing values: 0
Cleaned files saved successfully!


In [16]:
from src.enrichment import *

listings, calendar, reviews = load_data()

occupancy = calculate_occupancy(
    calendar
)

occupancy.head()

occupancy.to_csv(
    "../data/processed/occupancy.csv",
    index=False
)